LLMChain的使用

举例1：

In [2]:
from langchain_classic.chains import LLMChain
from langchain_classic.chains.sequential import SimpleSequentialChain
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, XMLOutputParser, JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
import dotenv

#1.创建大模型
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0.8,

)

#提供提示词模板
#BasePromptTemplate的典型子类有：PromptTemplate、ChatPromptTemplate
prompt_template = PromptTemplate.from_template(
    template="你是一个数学高手，帮我解决如下的数学问题：{question}"
)

chain = LLMChain(llm=chat_model, prompt=prompt_template)
response = chain.invoke(input={"question": "1+4*9=？"})
print(response)

{'question': '1+4*9=？', 'text': '根据数学运算的优先级，先进行乘法运算，然后再进行加法运算。\n\n所以，首先计算 \\(4 \\times 9\\):\n\n\\[ \n4 \\times 9 = 36 \n\\]\n\n然后将结果加上1：\n\n\\[ \n1 + 36 = 37 \n\\]\n\n因此， \\(1 + 4 \\times 9 = 37\\)。'}


2.顺序链之SimpleSequentialChain的使用

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

chainA_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一位精通各领域知识的知名教授"),
        ("human", "请你尽可能详细的解释一下：{knowledge}"),
    ]
)

chainA_chains = LLMChain(llm=chat_model,
                         prompt=chainA_template,
                         verbose=True
                         )

In [4]:
from langchain_core.prompts import ChatPromptTemplate

chainB_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你非常善于提取文本中的重要信息，并做出简短的总结"),
        ("human", "这是针对一个提问的完整的解释说明内容：{description}"),
        ("human", "请你根据上述说明，尽可能简短的输出重要的结论，请控制在20个字以内"),
    ]
)
chainB_chains = LLMChain(llm=chat_model,
                         prompt=chainB_template,
                         verbose=True
                         )

full_chain = SimpleSequentialChain(
    chains=[chainA_chains, chainB_chains],
    verbose=True
)

#说明：：针对于SimpleSequentialChain而言，唯一的输入变量名是：input
response = full_chain.invoke(input={"input": "什么是LangChain"})
print(response)



> Entering new SimpleSequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 你是一位精通各领域知识的知名教授
Human: 请你尽可能详细的解释一下：什么是LangChain

> Finished chain.
LangChain 是一个用于构建与语言模型（如 GPT-3、GPT-4）互动的应用程序的框架。它的目标是简化与大型语言模型的集成和使用，特别是在处理复杂任务时。LangChain 提供了一系列工具和组件，使开发者能够更容易地创建智能应用程序，这些应用程序能够理解和生成自然语言。

### LangChain 的核心组成部分

1. **链（Chains）**：
   - LangChain 的核心概念是“链”。链是由多个组件串联在一起的流程，这些组件可以是语言模型、数据源（如数据库或API）或其他处理步骤。通过将这些组件组合在一起，开发者可以创建复杂的工作流。

2. **提示（Prompts）**：
   - 在与语言模型交互时，提示（或输入文本）是至关重要的。LangChain 提供了一种灵活的提示模板机制，使开发者能够动态生成和管理提示，从而更好地控制模型的输出。

3. **记忆（Memory）**：
   - LangChain 支持记忆机制，使应用程序能够在会话中保持上下文。这对于构建需要上下文理解的对话系统尤其重要。记忆可以是短期的（仅在一次会话中有效）或长期的（跨越多个会话）。

4. **工具（Tools）**：
   - LangChain 允许开发者将外部工具和API集成到工作流中。这些工具可以是计算、数据查询、网络抓取等功能，使得语言模型能够在执行任务时访问额外的信息和功能。

5. **数据代理（Agents）**：
   - LangChain 支持创建代理，这些代理能够根据上下文和用户输入动态选择使用哪个工具或执行哪个操作。代理可以根据特定任务的需求，灵活地调整其行为。

### LangChain 的应用场景

- **对话系统**：构建能够进行自然对话的聊天机器人。
- **自动化任务**：创建自动回复邮件、生成报告或数据分析的工具。
- **内容创作**：帮助用户生

SequentialChain多个输入多个输出的使用

举例1

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import SequentialChain
from langchain_openai import ChatOpenAI
from langchain_classic.chains import LLMChain
from openai import OpenAI
import os

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")
schainA_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一位精通各领域知识的知名教授"),
        ("human", "请你先尽可能详细的解释一下：{knowledge}，并且{action}")
    ]
)
schainA_chains = LLMChain(llm=llm,
                          prompt=schainA_template,
                          verbose=True,
                          output_key="schainA_chains_key"
                          )
# schainA_chains.invoke({
# "knowledge": "中国的篮球怎么样？",
# "action": "举一个实际的例子"
# }
# )
schainB_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你非常善于提取文本中的重要信息，并做出简短的总结"),
        ("human", "这是针对一个提问完整的解释说明内容：{schainA_chains_key}"),
        ("human", "请你根据上述说明，尽可能简短的输出重要的结论，请控制在100个字以内"),
    ]
)
schainB_chains = LLMChain(llm=llm,
                          prompt=schainB_template,
                          verbose=True,
                          output_key='schainB_chains_key'
                          )

#一定要声明出两个变量：input_variables,output_variables
Seq_chain = SequentialChain(
    chains=[schainA_chains, schainB_chains],
    input_variables=["knowledge", "action"],
    output_variables=["schainA_chains_key", "schainB_chains_key"],
    verbose=True)
response = Seq_chain.invoke({
    "knowledge": "中国足球为什么踢得烂",
    "action": "举一个实际的例子"
}
)
print(response)



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 你是一位精通各领域知识的知名教授
Human: 请你先尽可能详细的解释一下：中国足球为什么踢得烂，并且举一个实际的例子

> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
System: 你非常善于提取文本中的重要信息，并做出简短的总结
Human: 这是针对一个提问完整的解释说明内容：中国足球的表现长期以来受到多种因素的影响，这些因素可以从文化、管理、青训、基础设施、政策等多个方面进行分析。

### 1. 历史和文化因素
中国的足球历史相对较短，尽管早在20世纪初期就有足球活动，但在很长一段时间内，足球并没有成为主流运动。传统文化对个人和集体利益的理解，以及对失败的宽容度，也使得足球等竞争性运动没有得到充分重视。

### 2. 管理体制
中国足球的管理体制相对复杂，足球协会在许多方面存在官僚主义和腐败问题，导致职业联赛的健康发展受到影响。此外，许多俱乐部的经营模式不合理，资金的运用和管理缺乏透明度，制约了球队的专业化水平。

### 3. 青训和人才培养
青训体系的不健全是中国足球的另一大问题。虽然近年来中国不少学校开始设置足球课程，但整体的青少年训练系统仍然不完善。很多年轻球员缺乏系统的训练和实战经验，导致在技术和战术理解上与国际水平存在差距。

### 4. 足球基础设施
尽管近年来中国在足球基础设施上进行了投资，但与足球强国相比，整体水平仍有差距。足球场地不足、训练设施落后等问题依然存在，影响了球员的日常训练和发展。

### 5. 政策环境
中国足球发展的政策环境也相对复杂。虽然国家层面曾出台多项政策以推动足球发展，如《国家足球发展战略》等，但实际执行过程中常常受到地方政府、俱乐部利益等多方面因素的影响，使得政策的落实效果大打折扣。

### 实际例子：2016年中国男足的“十二强赛”
在2016年亚洲世界杯预选赛的“十二强赛”中，中国男足的表现极为糟糕，最终仅以两场平局和八场失利的成绩排名末位。这场比赛中，中国队在对阵强队时显得无